# Macaulay2 in your browser — via the MCP server

This notebook runs the [**macaulay2-mcp**](https://github.com/youngsu-Kim/macaulay2-mcp) server and drives it with the standard [Model Context Protocol](https://modelcontextprotocol.io) — exactly the way Claude Code or opencode would on your own machine. No Macaulay2 or MCP installation needed here: everything (M2 1.26, the server, the client) is already inside this Binder session.

> Note: this is a **plain Python 3 notebook** — it does *not* use a Macaulay2 Jupyter kernel. The cells below are an MCP *client* talking to the MCP *server* over stdio.

Run the cells top to bottom. The session state (rings, ideals, ...) persists across cells, just like in the real tool.

> The notebook file is yours: **download** (File menu), **copy**, or **modify** it freely — it is GPL-3.0-or-later like the server, and this Binder session grants your user full write access to it.


In [ ]:
# Connect to the macaulay2 MCP server (one persistent session for the whole notebook)
#
# Jupyter kernels already run an asyncio loop, so we must NOT call
# run_until_complete() from a cell. Instead: one dedicated event loop on a
# background thread owns the MCP connection; every coroutine is submitted to
# it with run_coroutine_threadsafe().
import asyncio, threading
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

SERVER_COMMAND = ["uvx", "macaulay2-mcp"]

loop = asyncio.new_event_loop()
threading.Thread(target=loop.run_forever, daemon=True, name="mcp-loop").start()

async def _open():
    global _ctx, session
    _ctx = stdio_client(StdioServerParameters(command=SERVER_COMMAND[0], args=SERVER_COMMAND[1:]))
    read, write = await _ctx.__aenter__()
    session = ClientSession(read, write)
    await session.__aenter__()
    return await session.initialize()

info = asyncio.run_coroutine_threadsafe(_open(), loop).result(timeout=180)
print("Connected to:", getattr(info, "server_info", None) or getattr(info, "serverInfo", None))

In [ ]:
# Small helper: call an MCP tool and return its text output
def m2(tool, **args):
    async def _call():
        result = await session.call_tool(tool, args)
        return result.content[0].text
    return asyncio.run_coroutine_threadsafe(_call(), loop).result(timeout=600)

# What tools does the server expose?
async def _list():
    return await session.list_tools()
tools = asyncio.run_coroutine_threadsafe(_list(), loop).result(timeout=120)
print("\n".join(t.name + " — " + (t.description or "").splitlines()[0] for t in tools.tools))

## A classic example: Groebner basis and Betti table

Compute the Groebner basis of $I = (x^3 - y,\; x^4 - z) \subset \mathbb{Q}[x,y,z]$ and a graded free resolution with its Betti table.

In [ ]:
print(m2("m2_evaluate", code="R = QQ[x,y,z]\nI = ideal(x^3 - y, x^4 - z)"))

In [ ]:
print(m2("m2_evaluate", code="print generators (gb I)"))

In [ ]:
print(m2("m2_evaluate", code="G = res I\nbetti G"))

## More things to try

- Documentation lookup: `m2("m2_help", topic="HilbertPolynomial")`
- Primary decomposition: `m2("m2_evaluate", code="primaryDecomposition I")`
- Load a package: `m2("m2_load_package", name="BoijSoederberg")`
- Your own definitions: write a `.m2` file in this session (e.g. via `!cat > mydefs.m2`) and import it with `m2("m2_import_file", path="/home/jovyan/mydefs.m2")` (adjust the path to your Binder home).

## Use it on your own machine

This Binder session is a demo. For real work, install the server once and point your AI coding assistant at it — see the main repository's README:

https://github.com/youngsu-Kim/macaulay2-mcp